##Citibike Ridership Modeling

*Goal:* build, evaluate, and interpret a Linear Regression model predicting `num_rides` from calendar + weather-forecast features only.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(r'C:\Users\Work\Documents\GitHub\citibike-ridership\data\citibike_weather.csv')
df.head()

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,Monday,7
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,Tuesday,7
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,Wednesday,7
3,2013-07-04,22326,21.218474,82.0,91.0,73.9,8.6,0.96,Thursday,7
4,2013-07-05,21842,18.040443,84.4,93.0,75.9,9.0,0.00,Friday,7


#M1: Split the Data
Load the clean CSV and split into training and test sets with train_test_split (80/20, and set a random_state so your results are reproducible).
 Tip: Keep ride_date out of your feature matrix — it identifies the row, it isn't a feature. Your trend column from C4 carries the time information instead.



In [2]:
# build X, y and split
X = df[['trend_feature', 'stretch_feature']]
y = df['target_variable']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


KeyError: "None of [Index(['trend_feature', 'stretch_feature'], dtype='str')] are in the [columns]"

#M2: Fit the Core Model
Fit a LinearRegression on your core feature set: your chosen temperature feature, precipitation, wind, the day-of-week dummies, and your trend feature. Respect both Rules of the Road.



In [ ]:
# fit LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)


#M3: Evaluate
Report R² on both the training set and the test set. In a markdown cell: what does your test R² mean in plain English, and why do we care about the TEST number more than the training number?
Also, report your test error in real units: MAE and RMSE, in rides. In the same markdown cell, translate MAE into one sentence which ops could actually use (“our prediction is typically off by ___ rides.”) and compare RMSE to MAW. RMSE is always at least as large, and the wider the gap, the more your total error is dominated by a few badly-missed days. Which days do you suspect those are?



In [ ]:
# R2, MAE, RMSE
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")


#M4: Interpret the Coefficients
Print your model's coefficients next to their feature names, and translate the three most interesting ones into plain English for the ops team — e.g., “each additional inch of rain costs us roughly ___ rides.” Do the signs match what your EDA led you to expect? Flag any that surprise you.



In [ ]:
# coefficients
print("Model Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef}")
    

#M5: Diagnose With Residuals
Plot residuals (actual − predicted) two ways: against predicted values, and against ride_date. In a markdown cell, describe any patterns — clumps, curves, or drift. A residual plot with structure in it is the model telling you what feature it's missing.
 Tip: If your residuals drift upward or downward over the years, revisit C4 — that's exactly the symptom a trend feature cures.



In [ ]:
# residual plots
# residual plots
residuals = y_test - y_pred

# Align residuals to their original row dates and plot both residual views
residuals = pd.Series(residuals, index=y_test.index, name='residuals')
test_dates = pd.to_datetime(df.loc[y_test.index, 'ride_date'])

# Residuals vs. predicted rides
plt.figure(figsize=(10, 4))
plt.scatter(y_pred, residuals, alpha=0.7, color='tab:blue')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Predicted num_rides')
plt.ylabel('Actual - Predicted (residual)')
plt.title('Residuals vs Predicted Values')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Residuals vs. ride date
plt.figure(figsize=(12, 4))
plt.scatter(test_dates, residuals, alpha=0.7, color='tab:orange')
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Ride Date')
plt.ylabel('Actual - Predicted (residual)')
plt.title('Residuals vs Ride Date')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#M6: Improve One Thing
Based on your residual diagnosis, make ONE change — add a stretch feature from C5, swap your temperature choice, transform a feature — refit, and compare test R² before vs. after. Keep both results visible in the notebook, even if the change made things worse. Negative results are results.



In [ ]:
# refit with one change, compare


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('../data/citibike_weather_daily_clean.csv')

# M1: build X, y
# Excluded: ride_date (identifier, not a feature)
#           avg_duration_min (Rule 1 - not knowable in advance)
#           max_temp_f, min_temp_f (Rule 2 - correlated 0.95-0.98 with temp_f; keeping temp_f only)
exclude_cols = ['ride_date', 'num_rides', 'avg_duration_min', 'max_temp_f', 'min_temp_f']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['num_rides']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Feature columns:", X.columns.tolist())
print("X_train:", X_train.shape, " X_test:", X_test.shape)